# 📦 Notebook 1 — Tiền xử lý & Lưu Dữ liệu GPM
> **Chạy notebook này trên acc CPU thông thường (không cần GPU)**  
> Kết quả sẽ lưu file `.npy` vào Google Drive để Notebook 2 load lại.

### Luồng xử lý:
```
File .nc4 trên Drive  →  Đọc & cắt Việt Nam  →  Chuẩn hóa  →  Lưu .npy
```


## 1. Cài đặt & Mount Drive


In [ ]:
!pip install -q netCDF4 h5py xarray cftime
print("✅ Cài đặt xong")


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✅ Drive đã mount")


In [ ]:
import os, glob, warnings
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ================================================================
# CẤU HÌNH — Chỉnh lại cho đúng với Drive của em
# ================================================================
GPM_DATA_DIR  = "/content/drive/MyDrive/gpm_data"       # thư mục chứa *.nc4
SAVE_DIR      = "/content/drive/MyDrive/rainfall_processed"  # thư mục lưu kết quả

# Vùng Việt Nam
LAT_MIN, LAT_MAX = 8.0, 23.0
LON_MIN, LON_MAX = 102.0, 110.0
# ================================================================
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"✅ Source : {GPM_DATA_DIR}")
print(f"   Dest   : {SAVE_DIR}")


## 2. Đọc & Xử lý Dữ liệu GPM


In [ ]:
def load_gpm_file(filepath):
    """
    Đọc 1 file GPM .nc4, cắt vùng Việt Nam → array (H, W) float32 mm/hr.
    Tự động xử lý cả 2 thứ tự chiều: (time,lon,lat) và (time,lat,lon).
    """
    ds    = xr.open_dataset(filepath, engine='netcdf4')
    ds_vn = ds.sel(lat=slice(LAT_MIN, LAT_MAX),
                   lon=slice(LON_MIN, LON_MAX))

    dims = ds_vn["precipitation"].dims
    raw  = ds_vn["precipitation"].values   # (1, ?, ?)

    if dims == ('time', 'lon', 'lat'):
        rain = raw[0].T        # (lon,lat).T → (lat,lon)
    elif dims == ('time', 'lat', 'lon'):
        rain = raw[0]          # đã là (lat,lon)
    else:
        rain = raw.squeeze()
        if rain.shape[0] < rain.shape[1]:
            rain = rain.T      # đảm bảo lat > lon

    rain = np.where(rain < 0, 0.0, rain)
    rain = np.nan_to_num(rain, nan=0.0)
    ds.close()
    return rain.astype(np.float32)


# ── Đọc tất cả file ──────────────────────────────────────────
files = sorted(glob.glob(os.path.join(GPM_DATA_DIR, "*.nc4")))
if not files:
    raise FileNotFoundError(f"Không thấy .nc4 trong: {GPM_DATA_DIR}")
print(f"🔄 Tìm thấy {len(files)} files — bắt đầu tải...")

all_frames, bad = [], []
for i, f in enumerate(files):
    try:
        all_frames.append(load_gpm_file(f))
        if (i+1) % 100 == 0 or (i+1) == len(files):
            print(f"   [{i+1:>5}/{len(files)}] OK — shape mẫu: {all_frames[-1].shape}")
    except Exception as e:
        bad.append(os.path.basename(f))
        print(f"   ⚠️  Bỏ qua: {os.path.basename(f)}  ({e})")

rain_raw = np.array(all_frames, dtype=np.float32)   # (T, H, W)
T, H, W  = rain_raw.shape

print(f"\n✅ Đọc xong!")
print(f"   Shape   : {rain_raw.shape}  →  T={T}, H={H}, W={W}")
print(f"   Thời gian: {T} bước × 30 phút = {T*0.5:.0f} giờ")
print(f"   RAM      : {rain_raw.nbytes/1024**2:.1f} MB")
if bad:
    print(f"   Bỏ qua   : {len(bad)} files lỗi")


## 3. Kiểm tra Nhanh Dữ liệu Thô


In [ ]:
# Thống kê nhanh
print(f"Min / Max  : {rain_raw.min():.4f} / {rain_raw.max():.2f} mm/hr")
print(f"Mean       : {rain_raw.mean():.5f} mm/hr")
print(f"Có mưa     : {(rain_raw > 0.1).mean()*100:.2f}%")
print(f"Mưa nặng   : {(rain_raw > 10).mean()*100:.4f}%  (>10 mm/hr)")

# Vẽ nhanh bản đồ trung bình
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
ext = [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX]

im = axes[0].imshow(rain_raw.mean(0), cmap='Blues', origin='lower', aspect='auto', extent=ext)
plt.colorbar(im, ax=axes[0], label='mm/hr'); axes[0].set_title('Mưa trung bình')

im = axes[1].imshow(rain_raw.max(0),  cmap='hot_r', origin='lower', aspect='auto', extent=ext)
plt.colorbar(im, ax=axes[1], label='mm/hr'); axes[1].set_title('Mưa cực đại')

ts = rain_raw.mean((1,2))
axes[2].plot(ts[:500], lw=0.7, color='steelblue')
axes[2].fill_between(range(min(500,T)), ts[:min(500,T)], alpha=0.2, color='steelblue')
axes[2].set_title('Chuỗi thời gian (500 bước đầu)')
axes[2].set_xlabel('Bước ×30 phút'); axes[2].set_ylabel('mm/hr')

plt.suptitle('Kiểm tra dữ liệu GPM thô — Việt Nam', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_raw.png'), dpi=120, bbox_inches='tight')
plt.show()
print("✅ Lưu eda_raw.png")


## 4. Chuẩn hóa & Lưu File `.npy`


In [ ]:
# ── Chuẩn hóa: log1p + min-max ──────────────────────────────
# log1p xử lý phân bố lệch phải; min-max đưa về [0, 1]
rain_log = np.log1p(rain_raw)
LOG_MIN  = float(rain_log.min())
LOG_MAX  = float(rain_log.max())
rain_norm = ((rain_log - LOG_MIN) / (LOG_MAX - LOG_MIN + 1e-8)).astype(np.float32)

print(f"✅ Chuẩn hóa log1p + min-max:")
print(f"   log_min = {LOG_MIN:.6f}")
print(f"   log_max = {LOG_MAX:.6f}")
print(f"   norm range: [{rain_norm.min():.4f}, {rain_norm.max():.4f}]")

# ── Lưu ra Drive ──────────────────────────────────────────────
np.save(os.path.join(SAVE_DIR, "rain_norm.npy"),  rain_norm)   # dữ liệu chuẩn hóa
np.save(os.path.join(SAVE_DIR, "rain_raw.npy"),   rain_raw)    # giữ bản gốc để tham chiếu

# Lưu các hằng số chuẩn hóa (cần để denormalize sau này)
np.save(os.path.join(SAVE_DIR, "norm_params.npy"),
        np.array([LOG_MIN, LOG_MAX, LAT_MIN, LAT_MAX, LON_MIN, LON_MAX,
                  float(H), float(W), float(T)]))

# Lưu shape info dưới dạng text cho dễ đọc
with open(os.path.join(SAVE_DIR, "dataset_info.txt"), "w") as f:
    f.write(f"T={T}\nH={H}\nW={W}\n")
    f.write(f"LOG_MIN={LOG_MIN}\nLOG_MAX={LOG_MAX}\n")
    f.write(f"LAT_MIN={LAT_MIN}\nLAT_MAX={LAT_MAX}\n")
    f.write(f"LON_MIN={LON_MIN}\nLON_MAX={LON_MAX}\n")
    f.write(f"total_files={len(files)}\nbad_files={len(bad)}\n")

print(f"\n📦 Đã lưu vào: {SAVE_DIR}")
for fname in os.listdir(SAVE_DIR):
    fpath = os.path.join(SAVE_DIR, fname)
    size  = os.path.getsize(fpath) / 1024**2
    print(f"   {fname:<25}  {size:>8.1f} MB")

print(f"\n🎉 Notebook 1 HOÀN TẤT!")
print(f"   → Chuyển sang Notebook 2 (GPU) để huấn luyện mô hình.")
